# 01 · Coleta de Dados (DataSUS)

**Objetivo:** Baixar arquivos `.dbc` diretamente do servidor FTP público do DataSUS e convertê-los para `.csv`, alimentando as etapas seguintes do projeto.

**Inputs:** Nenhum (acesso direto ao FTP do DataSUS via internet).

**Outputs gerados:**
- `data/raw/SIH/` — arquivos `.dbc` brutos do Sistema de Informações Hospitalares
- `data/raw/CNES/` — arquivos `.dbc` brutos do Cadastro Nacional de Estabelecimentos
- `data/input/SIH/` — arquivos `.csv` convertidos do SIH
- `data/input/CNES/` — arquivos `.csv` convertidos do CNES

**Sistemas do DATASUS utilizados:**

| Sigla | Nome completo |
|-------|---------------|
| `SIH`  | Sistema de Informações Hospitalares |
| `CNES` | Cadastro Nacional de Estabelecimentos de Saúde |

<br/>

> **Nota:** O `SIM` (Sistema de Informação sobre Mortalidade) está disponível nos utilitários mas não é utilizado neste projeto.

## 0. Configuração do Ambiente

In [2]:
import sys
import os
import pandas as pd
from pathlib import Path

In [3]:
# Garante que a raiz do projeto está no sys.path para importar src/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /home/carolina/Documents/TCC Documentos/TCC


In [4]:
# Importa utilitários do projeto
from src.utils.download_data_from_datasus import download_data
from src.utils.converter_dbc_para_csv import converter_dbc_para_csv_lote

print("Utilitários importados")

Utilitários importados


## 1. Coleta do SIH (Sistema de Informações Hospitalares)

Baixamos os registros de AIH (Autorização de Internação Hospitalar) para o estado e ano definidos abaixo.
O fluxo é:
1. Download do `.dbc` do FTP do DataSUS → `data/raw/SIH/`
2. Conversão `.dbc → .csv` → `data/input/SIH/`

In [5]:
# ── Parâmetros de coleta ─────────────────────────────────────────────
ESTADOS_SIH = ["SP"]    # lista de UFs a baixar
ANOS_SIH    = [2025]    # anos de referência
SISTEMA_SIH = "SIH"     # identificador do sistema DataSUS

# ── Diretórios ───────────────────────────────────────────────────────
PASTA_RAW_SIH   = Path(ROOT, "data", "raw",   SISTEMA_SIH)
PASTA_INPUT_SIH = Path(ROOT, "data", "input", SISTEMA_SIH)

PASTA_RAW_SIH.mkdir(parents=True, exist_ok=True)
PASTA_INPUT_SIH.mkdir(parents=True, exist_ok=True)

print(f"RAW   : {PASTA_RAW_SIH}")
print(f"INPUT : {PASTA_INPUT_SIH}")

RAW   : /home/carolina/Documents/TCC Documentos/TCC/data/raw/SIH
INPUT : /home/carolina/Documents/TCC Documentos/TCC/data/input/SIH


In [7]:
# Download dos arquivos .dbc do SIH
print("Iniciando download do SIH...")
download_data(
    estados=ESTADOS_SIH,
    anos=ANOS_SIH,
    sistema=SISTEMA_SIH,
    download_path=str(PASTA_RAW_SIH),
)
print("Download concluído.")

Iniciando download do SIH...
Conectando ao FTP: ftp.datasus.gov.br

Download finalizado!
Arquivos salvos em: /home/carolina/Documents/TCC Documentos/TCC/data/raw/SIH
Download concluído.


In [8]:
# Conversão .dbc → .csv
print("Convertendo arquivos SIH de .dbc para .csv...")
converter_dbc_para_csv_lote(str(PASTA_RAW_SIH), str(PASTA_INPUT_SIH))

# Validação: lista arquivos gerados
csvs_sih = sorted(PASTA_INPUT_SIH.glob("*.csv"))
print(f"\nArquivos CSV gerados ({len(csvs_sih)}):")
for f in csvs_sih:
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

Convertendo arquivos SIH de .dbc para .csv...
Processando: RDSP2507.dbc
Concluído: rdsp2507.csv
Processando: RDSP2512.dbc
Concluído: rdsp2512.csv
Processando: RDSP2508.dbc
Concluído: rdsp2508.csv
Processando: RDSP2506.dbc
Concluído: rdsp2506.csv
Processando: RDSP2502.dbc
Concluído: rdsp2502.csv
Processando: RDSP2511.dbc
Concluído: rdsp2511.csv
Processando: RDSP2510.dbc
Concluído: rdsp2510.csv
Processando: RDSP2505.dbc
Concluído: rdsp2505.csv
Processando: RDSP2504.dbc
Concluído: rdsp2504.csv
Processando: RDSP2509.dbc
Concluído: rdsp2509.csv
Processando: RDSP2503.dbc
Concluído: rdsp2503.csv
Processando: RDSP2501.dbc
Concluído: rdsp2501.csv

Arquivos CSV gerados (12):
  rdsp2501.csv  (106533 KB)
  rdsp2502.csv  (103466 KB)
  rdsp2503.csv  (110689 KB)
  rdsp2504.csv  (110343 KB)
  rdsp2505.csv  (116755 KB)
  rdsp2506.csv  (110543 KB)
  rdsp2507.csv  (114801 KB)
  rdsp2508.csv  (114002 KB)
  rdsp2509.csv  (113085 KB)
  rdsp2510.csv  (115507 KB)
  rdsp2511.csv  (107749 KB)
  rdsp2512.csv  (1

## 2. Coleta do CNES (Cadastro Nacional de Estabelecimentos de Saúde)

Baixamos os arquivos das tabelas auxiliares do CNES:
- **ST** — Estabelecimentos (tabela principal)
- **LT** — Leitos
- **EQ** — Equipamentos
- **SR** — Serviços Especializados
- **HB** — Habilitações

Esses dados serão integrados com o SIH no notebook `02_eda.ipynb`.

In [ ]:
# ── Parâmetros de coleta ─────────────────────────────────────────────
ESTADOS_CNES = ["SP"]   # lista de UFs a baixar
ANOS_CNES    = [2025]   # anos de referência
MESES_CNES = [12]
BASES_CNES = ["ST","LT", "EQ", "HB", "SR"]
SISTEMA_CNES = "CNES"   # identificador do sistema DataSUS

# ── Diretórios ───────────────────────────────────────────────────────
PASTA_RAW_CNES   = Path(ROOT, "data", "raw",   SISTEMA_CNES)
PASTA_INPUT_CNES = Path(ROOT, "data", "input", SISTEMA_CNES)

PASTA_RAW_CNES.mkdir(parents=True, exist_ok=True)
PASTA_INPUT_CNES.mkdir(parents=True, exist_ok=True)

print(f"RAW   : {PASTA_RAW_CNES}")
print(f"INPUT : {PASTA_INPUT_CNES}")

RAW   : /home/carolina/Documents/TCC Documentos/TCC/data/raw/CNES
INPUT : /home/carolina/Documents/TCC Documentos/TCC/data/input/CNES


In [6]:
# Download dos arquivos .dbc do CNES
print("Iniciando download do CNES...")
download_data(
    estados=ESTADOS_CNES,
    anos=ANOS_CNES,
    meses=MESES_CNES,
    sistema=SISTEMA_CNES,
    bases_cnes=BASES_CNES,
    download_path=str(PASTA_RAW_CNES),
)
print("Download concluído.")

Iniciando download do CNES...
Conectando ao FTP: ftp.datasus.gov.br

Download finalizado!
Arquivos salvos em: /home/carolina/Documents/TCC Documentos/TCC/data/raw/CNES
Download concluído.


In [ ]:
# Conversão .dbc → .csv
print("Convertendo arquivos CNES de .dbc para .csv...")
converter_dbc_para_csv_lote(str(PASTA_RAW_CNES), str(PASTA_INPUT_CNES))


Convertendo arquivos CNES de .dbc para .csv...
Processando: LTSP2512.dbc
Concluído: ltsp2512.csv
Processando: EQSP2512.dbc
Concluído: eqsp2512.csv
Processando: HBSP2512.dbc
Concluído: hbsp2512.csv
Processando: SRSP2512.dbc
Concluído: srsp2512.csv
Processando: STSP2512.dbc
Erro na conversão: unpack requires a buffer of 32 bytes
Concluído: stsp2512.csv

Arquivos CSV gerados (4):
  eqsp2512.csv  (23958 KB)
  hbsp2512.csv  (1030 KB)
  ltsp2512.csv  (829 KB)
  srsp2512.csv  (19147 KB)


In [9]:
# Validação: lista arquivos gerados
csvs_cnes = sorted(PASTA_INPUT_CNES.glob("*.csv"))
print(f"\nArquivos CSV gerados ({len(csvs_cnes)}):")
for f in csvs_cnes:
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")


Arquivos CSV gerados (5):
  eqsp2512.csv  (23958 KB)
  hbsp2512.csv  (1030 KB)
  ltsp2512.csv  (829 KB)
  srsp2512.csv  (19147 KB)
  stsp2512.csv  (51484 KB)


## 3. Preview dos Dados Coletados

In [10]:
# Preview do primeiro arquivo SIH gerado
if csvs_sih:
    df_preview_sih = pd.read_csv(csvs_sih[0], nrows=5)
    print(f"SIH — {csvs_sih[0].name}: {df_preview_sih.shape[0]} linhas (amostra) × {df_preview_sih.shape[1]} colunas")
    display(df_preview_sih.head())
else:
    print("Nenhum arquivo SIH encontrado em", PASTA_INPUT_SIH)

SIH — rdsp2501.csv: 5 linhas (amostra) × 113 colunas


,UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,...,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9
0,350000,2025,1,1,46374500028366,3525100117847,1,11704840,354100,19840716,...,NaN,1,0,0,0,0,0,0,0,0
1,350000,2025,1,2,46374500028366,3524130275908,1,11741802,352210,20070606,...,NaN,1,1,1,1,1,0,0,0,0
2,350000,2025,1,2,46374500028366,3524130278427,1,11730000,353110,20030120,...,NaN,1,1,1,0,0,0,0,0,0
3,350000,2025,1,2,46374500028366,3524130278449,1,11743250,352210,20030405,...,NaN,1,1,1,1,0,0,0,0,0
4,350000,2025,1,2,46374500028366,3524130278526,1,11730000,353110,20050224,...,NaN,1,1,1,1,0,0,0,0,0


In [11]:
# Preview do primeiro arquivo CNES (ST) gerado
csvs_cnes_st = sorted(PASTA_INPUT_CNES.glob("st*.csv"))
if csvs_cnes_st:
    df_preview_cnes = pd.read_csv(csvs_cnes_st[0], nrows=5)
    print(f"CNES/ST — {csvs_cnes_st[0].name}: {df_preview_cnes.shape[0]} linhas (amostra) × {df_preview_cnes.shape[1]} colunas")
    display(df_preview_cnes.head())
else:
    print("Nenhum arquivo CNES/ST encontrado em", PASTA_INPUT_CNES)

CNES/ST — stsp2512.csv: 5 linhas (amostra) × 208 colunas


,CNES,CODUFMUN,COD_CEP,CPF_CNPJ,PF_PJ,NIV_DEP,CNPJ_MAN,COD_IR,REGSAUDE,MICR_REG,...,AP07CV02,AP07CV03,AP07CV04,AP07CV05,AP07CV06,AP07CV07,ATEND_PR,DT_ATUAL,COMPETEN,NAT_JUR
0,47406,350010,17800037,35723744000119,3,1,0,NaN,0209,NaN,...,0,0,0,0,0,0,1,202409,202512,2062
1,81655,350010,17800057,381929000108,3,1,0,NaN,R209,NaN,...,0,0,0,0,0,0,1,202508,202512,2062
2,109789,350010,17803116,36060657000191,3,1,0,NaN,0209,NaN,...,0,0,0,0,0,0,1,202409,202512,2135
3,109827,350010,17800005,48346595000168,3,1,0,NaN,0209,NaN,...,0,0,0,0,0,0,1,202409,202512,2135
4,183555,350010,17800043,36363624000110,3,1,0,NaN,0209,NaN,...,0,0,0,0,0,0,1,202410,202512,2135


ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Auxiliar/